In [1]:
import pandas as pd
import numpy as np

weather_all = pd.read_csv("../../data/processed/step_4-1_weather.csv")
weather_all["date"] = pd.to_datetime(weather_all["date"])
weather_all = weather_all.sort_values(["stn", "date"]).reset_index(drop=True)

weather_cols = ["ta_min", "ta_max", "rn_day", "rhm_avg", "ws_davg"]

# -99 → NaN
for col in weather_cols:
    weather_all[col] = weather_all[col].replace(-99, np.nan)

# 관측소별 이동 중앙값(15일) 대비 편차 계산
temp_cols = ["ta_min", "ta_max", "rhm_avg"]
for col in temp_cols:
    med = (
        weather_all.groupby("stn")[col]
        .transform(lambda x: x.rolling(15, center=True, min_periods=3).median())
    )
    weather_all[f"{col}_dev"] = weather_all[col] - med

# 편차 절대값 상위 15개 후보 확인
for col in temp_cols:
    print(f"\n===== {col} 편차 절대값 상위 15개 =====")
    idx = weather_all[f"{col}_dev"].abs().sort_values(ascending=False).index
    print(weather_all.loc[idx, ["stn", "date", col, f"{col}_dev"]].head(15).to_string())


===== ta_min 편차 절대값 상위 15개 =====
         stn       date  ta_min  ta_min_dev
719575   268 2021-07-01   -10.5      -32.25
1290697  725 2021-07-18     0.0      -25.40
1290698  725 2021-07-19     0.0      -25.40
1393720  781 2021-07-17     0.0      -25.40
1117909  609 2022-08-02     0.2      -25.20
907116   330 2021-08-06     0.0      -23.90
1393759  781 2021-08-25     0.0      -23.70
156942   129 2021-07-22     0.0      -23.20
156939   129 2021-07-19     0.0      -23.20
1185142  644 2020-08-13     0.0      -22.90
1240662  690 2022-08-03     2.4      -22.30
1593184  918 2021-07-09     0.0      -22.10
1139460  621 2021-07-29     0.0      -22.00
1413856  792 2022-08-29     0.0      -22.00
872398   302 2022-07-26     0.2      -22.00

===== ta_max 편차 절대값 상위 15개 =====
         stn       date  ta_max  ta_max_dev
995264   542 2022-11-17    43.4        26.3
1119188  611 2020-02-01    29.1        20.9
1613890  931 2024-03-13    32.3        19.0
1621160  936 2020-02-05    28.5        18.8
1528772 

In [2]:
# 의심 케이스 주변 날짜 확인
cases = [
    (268,  "2021-06-25", "2021-07-08"),   # ta_min -10.5 (7월)
    (725,  "2021-07-12", "2021-07-25"),   # ta_min 0.0 (7월)
    (611,  "2020-01-25", "2020-02-08"),   # ta_max 29.1 (2월)
    (931,  "2024-03-07", "2024-03-20"),   # ta_max 32.3 (3월)
    (625,  "2021-02-10", "2021-02-24"),   # ta_max -7.9 (2월 한파?)
    (152,  "1996-02-07", "1996-02-21"),   # ta_max 24.2 (2월, 편차 +18.0)
    (243,  "1996-02-06", "1996-02-20"),   # ta_max 19.7 (2월, 편차 +17.7)
    (285,  "1996-02-07", "1996-02-21"),   # ta_max 23.9 (2월, 편차 +17.5)
]

for stn, start, end in cases:
    print(f"\n===== stn {stn} {start} ~ {end} =====")
    tmp = weather_all[
        (weather_all["stn"] == stn) &
        (weather_all["date"] >= start) &
        (weather_all["date"] <= end)
        ]
    print(tmp[["date", "ta_min", "ta_max", "rhm_avg"]].to_string())


===== stn 268 2021-06-25 ~ 2021-07-08 =====
             date  ta_min  ta_max  rhm_avg
719569 2021-06-25     NaN     NaN     81.4
719570 2021-06-26     NaN     NaN     76.4
719571 2021-06-27     NaN     NaN     77.8
719572 2021-06-28     NaN     NaN     82.0
719573 2021-06-29     NaN     NaN     83.8
719574 2021-06-30     NaN     NaN     77.5
719575 2021-07-01   -10.5    28.6     75.9
719576 2021-07-02    19.5    29.2     81.0
719577 2021-07-03    22.5    26.0     89.5
719578 2021-07-04    21.0    25.7     87.4
719579 2021-07-05    18.3    25.3     96.8
719580 2021-07-06    23.9    26.2     95.0
719581 2021-07-07    24.9    27.4     92.9
719582 2021-07-08    23.7    27.9     95.1

===== stn 725 2021-07-12 ~ 2021-07-25 =====
              date  ta_min  ta_max  rhm_avg
1290691 2021-07-12    25.6    30.2     88.0
1290692 2021-07-13    25.2    29.7     91.0
1290693 2021-07-14    25.4    28.7     93.0
1290694 2021-07-15    24.7    30.9     86.0
1290695 2021-07-16    25.6    29.6     86.0
1

In [3]:
# 이상치 처리 (rolling median 편차 기반)
weather_cols_temp = ["ta_min", "ta_max"]

# rolling median 계산
for col in weather_cols_temp:
    med = (
        weather_all.groupby("stn")[col]
        .transform(lambda x: x.rolling(15, center=True, min_periods=3).median())
    )
    weather_all[f"{col}_med"] = med
    weather_all[f"{col}_dev"] = weather_all[col] - med

# 이상치 → NaN
# ta_min: 편차 < -20
ta_min_outliers = (weather_all["ta_min_dev"] < -20).sum()
weather_all.loc[weather_all["ta_min_dev"] < -20, "ta_min"] = np.nan

# ta_max: 편차 > +18
ta_max_outliers = (weather_all["ta_max_dev"] > 18).sum()
weather_all.loc[weather_all["ta_max_dev"] > 18, "ta_max"] = np.nan

print(f"ta_min 이상치 처리: {ta_min_outliers:,}개")
print(f"ta_max 이상치 처리: {ta_max_outliers:,}개")

# 임시 컬럼 제거
weather_all = weather_all.drop(columns=[
    "ta_min_med", "ta_min_dev", "ta_max_med", "ta_max_dev",
    "rhm_avg_dev"
])

# 처리 후 확인
print(f"\n처리 후 shape: {weather_all.shape}")
print(f"ta_min 결측: {weather_all['ta_min'].isnull().sum():,}")
print(f"ta_max 결측: {weather_all['ta_max'].isnull().sum():,}")

ta_min 이상치 처리: 26개
ta_max 이상치 처리: 4개

처리 후 shape: (1645237, 7)
ta_min 결측: 12,426
ta_max 결측: 12,403


In [4]:
import numpy as np

weather_cols = ["ta_min", "ta_max", "rn_day", "rhm_avg", "ws_davg"]
interp_cols  = ["ta_min", "ta_max", "rhm_avg", "ws_davg"]

# =====================================================
# 1. rn_day 결측 → 0 (강수 없는 날)
# =====================================================
weather_all["rn_day"] = weather_all["rn_day"].fillna(0)

# =====================================================
# 2. 관측소별 선형 보간 (연속 3일 이하)
# =====================================================
weather_all = weather_all.sort_values(["stn", "date"]).reset_index(drop=True)

weather_all[interp_cols] = (
    weather_all.groupby("stn")[interp_cols]
    .transform(lambda x: x.interpolate(method="linear", limit=3))
)

print("보간 후 결측:")
print(weather_all[interp_cols].isnull().sum())

# =====================================================
# 3. 보간 후 ta_min > ta_max 모순 → 둘 다 NaN
# =====================================================
contradiction = weather_all["ta_min"] > weather_all["ta_max"]
print(f"\nta_min > ta_max 모순: {contradiction.sum():,}개")
weather_all.loc[contradiction, ["ta_min", "ta_max"]] = np.nan

# =====================================================
# 4. 잔여 결측 → 관측소별 + 월별 중앙값
# =====================================================
weather_all["month"] = weather_all["date"].dt.month

weather_all[interp_cols] = (
    weather_all.groupby(["stn", "month"])[interp_cols]
    .transform(lambda x: x.fillna(x.median()))
)

print("\n월별 중앙값 후 결측:")
print(weather_all[interp_cols].isnull().sum())

# =====================================================
# 5. 그래도 남은 결측 → 관측소별 중앙값
# =====================================================
weather_all[interp_cols] = (
    weather_all.groupby("stn")[interp_cols]
    .transform(lambda x: x.fillna(x.median()))
)

print("\n관측소별 중앙값 후 결측:")
print(weather_all[interp_cols].isnull().sum())

# =====================================================
# 6. 최종 ta_min > ta_max 재확인
# =====================================================
final_contradiction = (weather_all["ta_min"] > weather_all["ta_max"]).sum()
print(f"\n최종 ta_min > ta_max 모순: {final_contradiction:,}개")

보간 후 결측:
ta_min     10936
ta_max     10934
rhm_avg    44689
ws_davg     5697
dtype: int64

ta_min > ta_max 모순: 0개

월별 중앙값 후 결측:
ta_min     0
ta_max     0
rhm_avg    0
ws_davg    0
dtype: int64

관측소별 중앙값 후 결측:
ta_min     0
ta_max     0
rhm_avg    0
ws_davg    0
dtype: int64

최종 ta_min > ta_max 모순: 0개


In [5]:
# month 컬럼 제거 후 저장
weather_all = weather_all.drop(columns=["month"])

weather_all.to_csv("../data/processed/step_4-2_weather.csv",
                   index=False, encoding="utf-8-sig")

print(f"저장 완료: {weather_all.shape}")
print(weather_all[weather_cols].isnull().sum())

저장 완료: (1645237, 7)
ta_min     0
ta_max     0
rn_day     0
rhm_avg    0
ws_davg    0
dtype: int64
